In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
!pip install qiskit qiskit-machine-learning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00


In [9]:
import os
normal_path = "/content/drive/MyDrive/chd_data/Normal"
print("Files in Normal directory:", os.listdir(normal_path))

Files in Normal directory: ['imgi_194_default.png', 'imgi_200_default.png', 'imgi_193_default.png', 'imgi_209_default.png', 'imgi_202_default.png', 'imgi_205_default.png', 'imgi_196_default.png', 'imgi_211_default.png', 'imgi_197_default.png', 'imgi_195_default.png', 'imgi_201_default.png', 'imgi_206_default.png', 'imgi_192_default.png', 'imgi_191_default.png', 'imgi_208_default.png', 'imgi_207_default.png', 'imgi_189_default.png', 'imgi_190_default.png', 'imgi_210_default.png', 'imgi_199_default.png', 'imgi_204_default.png', 'imgi_203_default.png', 'imgi_198_default.png']


In [10]:
import os
import shutil
import glob
from sklearn.model_selection import train_test_split

# Define base paths
RAW_DATA_DIR = "/content/drive/MyDrive/chd_data"  # Expected subfolders: 'CHD' and 'Normal'
BASE_DIR = "dataset_split"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
AUG_TRAIN_DIR = os.path.join(BASE_DIR, "augmented_train")

CLASSES = ["CHD", "Normal"]

# Create output directories
for folder in [TRAIN_DIR, TEST_DIR, AUG_TRAIN_DIR]:
    for cls in CLASSES:
        os.makedirs(os.path.join(folder, cls), exist_ok=True)

# Collect image file paths and labels
filepaths = []
labels = []

valid_exts = (
    "*.png", "*.PNG",
    "*.jpg", "*.JPG",
    "*.jpeg", "*.JPEG",
    "*.bmp", "*.BMP",
    "*.tif", "*.TIF",
    "*.tiff", "*.TIFF"
)

for cls in CLASSES:
    cls_path = os.path.join(RAW_DATA_DIR, cls)
    images = []
    for ext in valid_exts:
        images.extend(glob.glob(os.path.join(cls_path, ext)))

    print(f"Found {len(images)} images in class '{cls}'")
    for img_path in images:
        filepaths.append(img_path)
        labels.append(cls)

print(f"\nTotal raw images: {len(filepaths)}")

Found 24 images in class 'CHD'
Found 23 images in class 'Normal'

Total raw images: 47


In [11]:
# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    filepaths,
    labels,
    test_size=0.20,
    stratify=labels,
    random_state=42
)

def copy_files(file_list, label_list, destination_dir):
    for src_path, label in zip(file_list, label_list):
        filename = os.path.basename(src_path)
        dst_path = os.path.join(destination_dir, label, filename)
        shutil.copy(src_path, dst_path)

# Copy split files to designated directories
copy_files(X_train, y_train, TRAIN_DIR)
copy_files(X_test, y_test, TEST_DIR)

print(f"Training set count: {len(X_train)} (CHD: {y_train.count('CHD')}, Normal: {y_train.count('Normal')})")
print(f"Test set count: {len(X_test)} (CHD: {y_test.count('CHD')}, Normal: {y_test.count('Normal')})")

Training set count: 37 (CHD: 19, Normal: 18)
Test set count: 10 (CHD: 5, Normal: 5)


In [12]:
import cv2
import numpy as np

def augment_image(image):
    """Applies domain-safe augmentations for fetal echocardiograms."""
    h, w = image.shape[:2]
    aug = image.copy()

    # 1. Random Horizontal Flip (50% probability)
    if np.random.rand() > 0.5:
        aug = cv2.flip(aug, 1)

    # 2. Random Mild Rotation (-10 to +10 degrees)
    angle = np.random.uniform(-10, 10)
    M_rot = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    aug = cv2.warpAffine(aug, M_rot, (w, h), borderMode=cv2.BORDER_REFLECT)

    # 3. Random Mild Scaling (0.95x to 1.05x)
    scale = np.random.uniform(0.95, 1.05)
    M_scale = cv2.getRotationMatrix2D((w // 2, h // 2), 0, scale)
    aug = cv2.warpAffine(aug, M_scale, (w, h), borderMode=cv2.BORDER_REFLECT)

    # 4. Random Brightness / Contrast Adjustment (±10%)
    alpha = np.random.uniform(0.9, 1.1)  # Contrast
    beta = np.random.uniform(-15, 15)     # Brightness
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)

    # 5. Low-variance Speckle Noise Injection
    if np.random.rand() > 0.5:
        gauss = np.random.normal(0, 0.03, aug.shape).astype(np.float32)
        noisy = aug.astype(np.float32) / 255.0 + gauss
        noisy = np.clip(noisy, 0, 1) * 255.0
        aug = noisy.astype(np.uint8)

    return aug

def balance_and_augment_training_set(train_dir, output_dir):
    # Copy all original training images first
    for cls in CLASSES:
        src_cls_dir = os.path.join(train_dir, cls)
        dst_cls_dir = os.path.join(output_dir, cls)
        for fname in os.listdir(src_cls_dir):
            shutil.copy(os.path.join(src_cls_dir, fname), os.path.join(dst_cls_dir, fname))

    # Calculate class counts
    counts = {cls: len(os.listdir(os.path.join(output_dir, cls))) for cls in CLASSES}
    max_count = max(counts.values())

    print(f"Initial Training Counts: {counts}")

    for cls, count in counts.items():
        if count < max_count:
            needed = max_count - count
            print(f"Augmenting '{cls}' by generating {needed} synthetic images...")

            existing_files = [
                os.path.join(output_dir, cls, f)
                for f in os.listdir(os.path.join(output_dir, cls))
            ]

            aug_idx = 0
            while aug_idx < needed:
                src_path = np.random.choice(existing_files)
                img = cv2.imread(src_path)
                if img is None:
                    continue

                aug_img = augment_image(img)
                out_name = f"aug_{aug_idx}_{os.path.basename(src_path)}"
                cv2.imwrite(os.path.join(output_dir, cls, out_name), aug_img)
                aug_idx += 1

    final_counts = {cls: len(os.listdir(os.path.join(output_dir, cls))) for cls in CLASSES}
    print(f"Final Balanced Training Counts: {final_counts}")

# Run balancing
balance_and_augment_training_set(TRAIN_DIR, AUG_TRAIN_DIR)

Initial Training Counts: {'CHD': 19, 'Normal': 18}
Augmenting 'Normal' by generating 1 synthetic images...
Final Balanced Training Counts: {'CHD': 19, 'Normal': 19}


In [13]:
import cv2
import numpy as np
import os
import glob
from skimage.segmentation import chan_vese

def preprocess_and_segment(image_path):
    """
    Executes Bilateral Filtering, CLAHE, and Chan-Vese Active Contour Segmentation.
    """
    # 1. Read image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, None, None

    # Resize to standard dimension for consistent feature scale (e.g., 256x256)
    img = cv2.resize(img, (256, 256))

    # 2. Bilateral Filter for noise suppression preserving edges
    bilateral = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)

    # 3. CLAHE Contrast Enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(bilateral)

    # 4. Chan-Vese Active Contour Segmentation
    # Scale image to float [0, 1] for scikit-image
    enhanced_norm = enhanced.astype(np.float32) / 255.0

    cv_mask = chan_vese(
        enhanced_norm,
        mu=0.25,
        lambda1=1.0,
        lambda2=1.0,
        tol=1e-3,
        max_num_iter=150,
        dt=0.5
    )

    # Convert mask to binary uint8 (0 or 255)
    segmented_img = (cv_mask.astype(np.uint8)) * 255

    return img, enhanced, segmented_img

In [26]:
def transform_test_set(test_dir, classes, kmeans_model, k_clusters=100):
    X_test_features = []
    y_test_labels = []
    test_image_paths = []  # Collect test image paths

    for cls in classes:
        cls_dir = os.path.join(test_dir, cls)
        for fname in os.listdir(cls_dir):
            path = os.path.join(cls_dir, fname)
            _, _, segmented = preprocess_and_segment(path)
            if segmented is None:
                continue

            desc = extract_sift_descriptors(segmented)
            histogram = np.zeros(k_clusters)

            if desc is not None:
                words = kmeans_model.predict(desc)
                for w in words:
                    histogram[w] += 1
                norm = np.linalg.norm(histogram)
                if norm > 0:
                    histogram = histogram / norm

            X_test_features.append(histogram)
            y_test_labels.append(1 if cls == "CHD" else 0)
            test_image_paths.append(path)  # Append test image path

    return np.array(X_test_features), np.array(y_test_labels), test_image_paths

In [19]:
from sklearn.cluster import KMeans

def extract_sift_descriptors(image):
    """Extracts SIFT keypoints and 128D descriptors."""
    sift = cv2.SIFT_create()
    keypoints, descriptors = sift.detectAndCompute(image, None)
    return descriptors

def build_codebook_and_features(data_dir, classes, k_clusters=100):
    all_descriptors = []
    image_descriptors_list = []
    labels = []
    image_paths = [] # Collect image paths

    print("Extracting SIFT descriptors from training dataset...")
    for cls in classes:
        cls_dir = os.path.join(data_dir, cls)
        for fname in os.listdir(cls_dir):
            path = os.path.join(cls_dir, fname)
            _, _, segmented = preprocess_and_segment(path)

            if segmented is None:
                continue

            desc = extract_sift_descriptors(segmented)
            if desc is not None:
                all_descriptors.append(desc)
                image_descriptors_list.append(desc)
                labels.append(1 if cls == "CHD" else 0)
                image_paths.append(path) # Append image path here

    # Stack all descriptors to form global dictionary space
    all_descriptors_stacked = np.vstack(all_descriptors)
    print(f"Total SIFT descriptors extracted: {all_descriptors_stacked.shape[0]}")

    # Cluster keypoints into K visual words
    print(f"Clustering descriptors into K={k_clusters} visual words via K-Means...")
    kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
    kmeans.fit(all_descriptors_stacked)

    # Map each image's descriptors to a K-dimensional histogram
    print("Constructing Bag-of-Visual-Words feature vectors...")
    X_features = []
    for descs in image_descriptors_list:
        histogram = np.zeros(k_clusters)
        words = kmeans.predict(descs)
        for w in words:
            histogram[w] += 1
        # L2 Normalization
        norm = np.linalg.norm(histogram)
        if norm > 0:
            histogram = histogram / norm
        X_features.append(histogram)

    X_features = np.array(X_features)
    y_labels = np.array(labels)

    return X_features, y_labels, kmeans, image_paths # Return image_paths

In [27]:
# Debugging transform_test_set return values
print("--- Debugging transform_test_set ---")

# Re-running build_codebook_and_features to ensure kmeans_model is available
# Note: This will re-process training data and might take some time.
# The variables AUG_TRAIN_DIR, CLASSES, k_clusters=100 are assumed to be defined in previous cells.
_, _, kmeans_model_debug, _ = build_codebook_and_features(AUG_TRAIN_DIR, CLASSES, k_clusters=100)

# Call transform_test_set and capture its output
# The variables TEST_DIR, CLASSES, k_clusters=100 are assumed to be defined in previous cells.
transformed_output = transform_test_set(TEST_DIR, CLASSES, kmeans_model_debug, k_clusters=100)

print(f"The transform_test_set function returned {len(transformed_output)} values.")
if len(transformed_output) != 3:
    print("ERROR: Expected 3 return values (X_test_features, y_test_labels, test_image_paths), but got a different number.")
else:
    print("SUCCESS: transform_test_set returned the expected 3 values.")
    print(f"  X_test_features shape: {transformed_output[0].shape}")
    print(f"  y_test_labels shape: {transformed_output[1].shape}")
    print(f"  test_image_paths count: {len(transformed_output[2])}")

print("------------------------------------")

--- Debugging transform_test_set ---
Extracting SIFT descriptors from training dataset...
Total SIFT descriptors extracted: 6643
Clustering descriptors into K=100 visual words via K-Means...
Constructing Bag-of-Visual-Words feature vectors...
The transform_test_set function returned 3 values.
SUCCESS: transform_test_set returned the expected 3 values.
  X_test_features shape: (10, 100)
  y_test_labels shape: (10,)
  test_image_paths count: 10
------------------------------------


In [47]:
import numpy as np
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from qiskit_machine_learning.algorithms import QSVC
from qiskit.circuit.library import zz_feature_map # Changed from ZZFeatureMap class to zz_feature_map function

# ==========================================
# 1. EXTRACT IMAGE FEATURES & LABELS
# ==========================================
print("--- Extracting BoVW Features ---")
X_train_img, y_train, kmeans_model, train_paths = build_codebook_and_features(AUG_TRAIN_DIR, CLASSES, k_clusters=100)
X_test_img, y_test, test_paths = transform_test_set(TEST_DIR, CLASSES, kmeans_model, k_clusters=100)

print(f"\nTraining Feature Shape: {X_train_img.shape}")
print(f"Test Feature Shape: {X_test_img.shape}")
print(f"Training Class Distribution: Normal = {np.sum(y_train == 0)}, CHD = {np.sum(y_train == 1)}")

# ==========================================
# 2. FEATURE SELECTION (TOP 12 FOR QUANTUM MAP)
# ==========================================
NUM_FEATURES = 12
selector = SelectKBest(score_func=f_classif, k=NUM_FEATURES)
X_train_selected = selector.fit_transform(X_train_img, y_train)
X_test_selected = selector.transform(X_test_img)

print(f"\nSelected top {NUM_FEATURES} SIFT features for SVM & QSVM.")

# ==========================================
# 3. TRAIN & EVALUATE CLASSICAL SVM
# ==========================================
print("\n--- Training Classical SVM ---")
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_model.fit(X_train_selected, y_train)

y_pred_svm = svm_model.predict(X_test_selected)
y_prob_svm = svm_model.predict_proba(X_test_selected)[:, 1]

print("\n[Classical SVM Results]")
print(classification_report(y_test, y_pred_svm, target_names=["Normal", "CHD"]))
print(f"AUC-ROC Score: {roc_auc_score(y_test, y_prob_svm):.4f}")

# ==========================================
# 4. TRAIN & EVALUATE QUANTUM SVM (QSVC)
# ==========================================
print("\n--- Training Quantum SVM (QSVC) ---")
feature_map = zz_feature_map(feature_dimension=NUM_FEATURES, reps=2) # Using the function instead of the class
qsvm_model = QSVC(feature_map=feature_map) # Corrected parameter name from quantum_kernel_feature_map to feature_map

qsvm_model.fit(X_train_selected, y_train)
y_pred_qsvm = qsvm_model.predict(X_test_selected)

print("\n[Quantum SVM Results]")
print(classification_report(y_test, y_pred_qsvm, target_names=["Normal", "CHD"]))
print(f"QSVM Accuracy: {accuracy_score(y_test, y_pred_qsvm):.4f}")

--- Extracting BoVW Features ---
Extracting SIFT descriptors from training dataset...
Total SIFT descriptors extracted: 6643
Clustering descriptors into K=100 visual words via K-Means...
Constructing Bag-of-Visual-Words feature vectors...

Training Feature Shape: (38, 100)
Test Feature Shape: (10, 100)
Training Class Distribution: Normal = 19, CHD = 19

Selected top 12 SIFT features for SVM & QSVM.

--- Training Classical SVM ---

[Classical SVM Results]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00         5
         CHD       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

AUC-ROC Score: 1.0000

--- Training Quantum SVM (QSVC) ---

[Quantum SVM Results]
              precision    recall  f1-score   support

      Normal       0.62      1.00      0.77         5
         CHD       1.00